In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

Reproductibilité



In [3]:
np.random.seed(42)
tf.random.set_seed(42)

FILE = "/content/donnees_IA_finales.csv.gz"

TARGET = "Pourcentage_validations"

Style graphiques

In [4]:
plt.rcParams.update({
    "figure.facecolor": "#0f1117", "axes.facecolor": "#1c1f2e",
    "axes.edgecolor": "#3a3f5c",   "axes.labelcolor": "#c9d1e0",
    "xtick.color": "#c9d1e0",      "ytick.color": "#c9d1e0",
    "text.color": "#c9d1e0",       "grid.color": "#2a2f45",
    "grid.linestyle": "--",        "grid.alpha": 0.5,
    "font.family": "DejaVu Sans",  "axes.titlesize": 13,
})
PALETTE = ["#5e8ef7", "#f7875e", "#5ef7a4", "#f7d35e"]

1. CHARGEMENT & FEATURE ENGINEERING

In [5]:
df = pd.read_csv(FILE, sep=";", compression="gzip", low_memory=False)

df["Date_Heure"] = pd.to_datetime(df["Date_Heure"], errors="coerce")
df = df.drop_duplicates()
df = df.sort_values("Date_Heure")
df["Temperature"] = df["Temperature"].interpolate(method="linear")
df["Pluie_1h"]    = df["Pluie_1h"].fillna(0)
df["ID_ZDC"]      = df["ID_ZDC"].fillna(-1)

Suppression outliers cible

In [6]:
q999 = df[TARGET].quantile(0.999)
df   = df[df[TARGET] <= q999]

Features temporelles

In [7]:
df["heure"]        = df["Date_Heure"].dt.hour
df["mois"]         = df["Date_Heure"].dt.month
df["jour_semaine"] = df["Date_Heure"].dt.dayofweek
df["est_weekend"]  = df["jour_semaine"].isin([5, 6]).astype(int)

Vacances scolaires zone C

In [8]:
vacances = [
    ("2025-02-15","2025-03-03"), ("2025-04-19","2025-05-05"),
    ("2025-07-05","2025-09-01"), ("2025-10-18","2025-11-03"),
    ("2025-12-20","2026-01-05"), ("2026-02-14","2026-03-02"),
    ("2026-04-18","2026-05-04"),
]
def is_vac(d):
    for a, b in vacances:
        if pd.Timestamp(a) <= d <= pd.Timestamp(b): return 1
    return 0
df["est_vacances"] = df["Date_Heure"].apply(is_vac)

Météo

In [9]:
df["il_pleut"]      = (df["Pluie_1h"] > 0).astype(int)
df["pluie_intense"] = (df["Pluie_1h"] > 2).astype(int)

Encodage cyclique

In [10]:
df["heure_sin"] = np.sin(2 * np.pi * df["heure"] / 24)
df["heure_cos"] = np.cos(2 * np.pi * df["heure"] / 24)
df["mois_sin"]  = np.sin(2 * np.pi * df["mois"] / 12)
df["mois_cos"]  = np.cos(2 * np.pi * df["mois"] / 12)
df["js_sin"]    = np.sin(2 * np.pi * df["jour_semaine"] / 7)
df["js_cos"]    = np.cos(2 * np.pi * df["jour_semaine"] / 7)

Encodage catégoriel

In [11]:
le_cat  = LabelEncoder()
le_arrt = LabelEncoder()
df["cat_jour_enc"] = le_cat.fit_transform(df["CAT_JOUR"])
df["arret_enc"]    = le_arrt.fit_transform(df["LIBELLE_ARRET"])

print(f"   Dataset prêt : {len(df):,} lignes")

   Dataset prêt : 8,561,208 lignes


###2 Préparation des features


In [12]:
FEATURES = [
    "heure_sin", "heure_cos",
    "mois_sin",  "mois_cos",
    "js_sin",    "js_cos",
    "Temperature", "Pluie_1h", "il_pleut", "pluie_intense",
    "est_weekend", "est_vacances",
    "cat_jour_enc", "arret_enc",
]

X = df[FEATURES].values
y = df[TARGET].values

Découpage temporel en 80/10/10 (restepecte ordre chronologique)

In [13]:
n = len(X)
n_test = int(n * 0.10)
n_val  = int(n * 0.10)

X_train, y_train = X[:n - n_test - n_val], y[:n - n_test - n_val]
X_val,   y_val   = X[n - n_test - n_val : n - n_test], y[n - n_test - n_val : n - n_test]
X_test,  y_test  = X[n - n_test:], y[n - n_test:]

Normalisation

In [14]:
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

print(f"\nSplits :")
print(f"Train : {len(X_train):,}  |  Val : {len(X_val):,}  |  Test : {len(X_test):,}")


Splits :
Train : 6,848,968  |  Val : 856,120  |  Test : 856,120


###3. Architecture MLP

In [15]:
def build_mlp(input_dim, learning_rate=1e-3):
    inp = keras.Input(shape=(input_dim,), name="input")

    x = layers.Dense(256, name="dense_1")(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Dense(128, name="dense_2")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Dense(64, name="dense_3")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(0.1)(x)

    x = layers.Dense(32, activation="relu", name="dense_4")(x)

    out = layers.Dense(1, name="output")(x)
    model = keras.Model(inputs=inp, outputs=out)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="huber",
        metrics=["mae"]
    )
    return model

model = build_mlp(len(FEATURES))
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input (InputLayer)              │ (None, 14)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │         3,840 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 48,897 (191.00 KB)

 Trainable params: 48,001 (187.50 KB)

 Non-trainable params: 896 (3.50 KB)

### 4. Entrainement

In [ ]:
print("\n Entraînement du MLP")
cb_list = [
    callbacks.EarlyStopping(
        monitor="val_loss", patience=10,
        restore_best_weights=True, verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5,
        patience=5, min_lr=1e-6, verbose=1
    ),
    callbacks.ModelCheckpoint(
        "best_mlp.keras", monitor="val_loss",
        save_best_only=True, verbose=0
    ),
]

print("\nEntraînement en cours")
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,              #Réduit
    batch_size=8192,        #Augmenté
    callbacks=cb_list,
    verbose=1,
)


 Entraînement du MLP

Entraînement en cours
Epoch 1/50
837/837 ━━━━━━━━━━━━━━━━━━━━ 114s 132ms/step - loss: 1.3601 - mae: 1.7382 - val_loss: 1.2468 - val_mae: 1.6101 - learning_rate: 0.0010
Epoch 2/50
837/837 ━━━━━━━━━━━━━━━━━━━━ 105s 126ms/step - loss: 1.2651 - mae: 1.6358 - val_loss: 1.2268 - val_mae: 1.5933 - learning_rate: 0.0010
Epoch 3/50
837/837 ━━━━━━━━━━━━━━━━━━━━ 143s 127ms/step - loss: 1.2427 - mae: 1.6112 - val_loss: 1.2141 - val_mae: 1.5790 - learning_rate: 0.0010
Epoch 4/50
837/837 ━━━━━━━━━━━━━━━━━━━━ 148s 134ms/step - loss: 1.2311 - mae: 1.5979 - val_loss: 1.2091 - val_mae: 1.5742 - learning_rate: 0.0010
Epoch 5/50
837/837 ━━━━━━━━━━━━━━━━━━━━ 138s 129ms/step - loss: 1.2255 - mae: 1.5913 - val_loss: 1.2079 - val_mae: 1.5732 - learning_rate: 0.0010
Epoch 6/50
837/837 ━━━━━━━━━━━━━━━━━━━━ 106s 126ms/step - loss: 1.2208 - mae: 1.5858 - val_loss: 1.2048 - val_mae: 1.5693 - learning_rate: 0.0010
Epoch 7/50
837/837 ━━━━━━━━━━━━━━━━━━━━ 109s 129ms/step - loss: 1.2174 - mae: 1

###5. Evaluation

In [17]:
print("\n Évaluation sur le jeu de test")

y_pred = model.predict(X_test, batch_size=4096, verbose=0).flatten()

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print(f"""
        RÉSULTATS
  MAE  : {mae:>8.4f}
  RMSE : {rmse:>8.4f}
  R²   : {r2:>8.4f}
""")


 Évaluation sur le jeu de test

        RÉSULTATS    
  MAE  :   1.5358              
  RMSE :   2.8432              
  R²   :   0.6062              



###6. Validation croisé

In [18]:
print("Validation croisée (5-Fold) sur échantillon 200k lignes")

SAMPLE = min(200_000, len(X_train))
idx    = np.random.choice(len(X_train), SAMPLE, replace=False)
Xc, yc = X_train[idx], y_train[idx]

kf      = KFold(n_splits=5, shuffle=True, random_state=42)
cv_maes, cv_r2s = [], []

for fold, (tr, va) in enumerate(kf.split(Xc), 1):
    m = build_mlp(len(FEATURES))
    m.fit(Xc[tr], yc[tr],
          validation_data=(Xc[va], yc[va]),
          epochs=30, batch_size=2048,
          callbacks=[callbacks.EarlyStopping(patience=5, restore_best_weights=True)],
          verbose=0)
    pv = m.predict(Xc[va], verbose=0).flatten()
    cv_maes.append(mean_absolute_error(yc[va], pv))
    cv_r2s.append(r2_score(yc[va], pv))
    print(f"   Fold {fold} → MAE={cv_maes[-1]:.4f}  R²={cv_r2s[-1]:.4f}")

print(f"\nCV MAE moyen : {np.mean(cv_maes):.4f} ± {np.std(cv_maes):.4f}")
print(f"CV R²  moyen : {np.mean(cv_r2s):.4f} ± {np.std(cv_r2s):.4f}")

Validation croisée (5-Fold) sur échantillon 200k lignes
   Fold 1 → MAE=1.6156  R²=0.5869
   Fold 2 → MAE=1.6212  R²=0.5771
   Fold 3 → MAE=1.6202  R²=0.5760
   Fold 4 → MAE=1.6074  R²=0.5811
   Fold 5 → MAE=1.6003  R²=0.5866

CV MAE moyen : 1.6129 ± 0.0080
CV R²  moyen : 0.5815 ± 0.0045


### 7. Graphique

In [20]:
def save_fig(fig, name):
    fig.savefig(name, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close(fig)
    print(f"✓ {name}")

7.1 Courbe Apprentissage

In [21]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Courbes d'apprentissage du MLP", fontweight="bold", color="#fff")

axes[0].plot(history.history["loss"],     color=PALETTE[0], label="Train", linewidth=2)
axes[0].plot(history.history["val_loss"], color=PALETTE[1], label="Val",   linewidth=2)
axes[0].set_xlabel("Époque"); axes[0].set_ylabel("Loss (Huber)")
axes[0].set_title("Loss"); axes[0].legend(); axes[0].grid(True)

axes[1].plot(history.history["mae"],     color=PALETTE[0], label="Train", linewidth=2)
axes[1].plot(history.history["val_mae"], color=PALETTE[1], label="Val",   linewidth=2)
axes[1].set_xlabel("Époque"); axes[1].set_ylabel("MAE")
axes[1].set_title("MAE"); axes[1].legend(); axes[1].grid(True)

fig.tight_layout()
save_fig(fig, "mlp_1_learning_curves.png")

✓ mlp_1_learning_curves.png


7.2 Prédiction vs Réel

In [22]:
PLOT_N = 5000
idx_s  = np.random.choice(len(y_test), PLOT_N, replace=False)

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(y_test[idx_s], y_pred[idx_s], alpha=0.2, s=8, color=PALETTE[0])
mn = min(y_test.min(), y_pred.min())
mx = max(y_test.max(), y_pred.max())
ax.plot([mn, mx], [mn, mx], color=PALETTE[1], linewidth=2, label="Parfait")
ax.set_xlabel("Valeurs réelles"); ax.set_ylabel("Valeurs prédites")
ax.set_title("Prédit vs Réel", fontweight="bold", color="#fff")
ax.legend(); ax.grid(True)
ax.text(0.05, 0.92, f"R² = {r2:.4f}\nMAE = {mae:.4f}", transform=ax.transAxes,
        color="#fff", fontsize=11, bbox=dict(boxstyle="round", facecolor="#2a2f45", alpha=0.8))
fig.tight_layout()
save_fig(fig, "mlp_2_pred_vs_reel.png")

✓ mlp_2_pred_vs_reel.png


7.3 Distrib des résidus

In [23]:
residus = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Analyse des résidus", fontweight="bold", color="#fff")

axes[0].hist(residus, bins=100, color=PALETTE[0], edgecolor="none", alpha=0.85)
axes[0].axvline(0, color=PALETTE[1], linewidth=2, linestyle="--")
axes[0].set_xlabel("Résidu (réel − prédit)"); axes[0].set_ylabel("Fréquence")
axes[0].set_title("Distribution des résidus"); axes[0].grid(True, axis="y")

axes[1].scatter(y_pred[idx_s], residus[idx_s], alpha=0.15, s=6, color=PALETTE[2])
axes[1].axhline(0, color=PALETTE[1], linewidth=1.5, linestyle="--")
axes[1].set_xlabel("Valeurs prédites"); axes[1].set_ylabel("Résidus")
axes[1].set_title("Résidus vs Prédictions"); axes[1].grid(True)

fig.tight_layout()
save_fig(fig, "mlp_3_residus.png")

✓ mlp_3_residus.png


7.4 Résultats validation croisé

In [24]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Validation croisée — 5 Folds", fontweight="bold", color="#fff")

folds = [f"Fold {i}" for i in range(1, 6)]
axes[0].bar(folds, cv_maes, color=PALETTE[0], alpha=0.85)
axes[0].axhline(np.mean(cv_maes), color=PALETTE[1], linestyle="--",
                linewidth=2, label=f"Moy = {np.mean(cv_maes):.4f}")
axes[0].set_ylabel("MAE"); axes[0].set_title("MAE par fold")
axes[0].legend(); axes[0].grid(True, axis="y")

axes[1].bar(folds, cv_r2s, color=PALETTE[2], alpha=0.85)
axes[1].axhline(np.mean(cv_r2s), color=PALETTE[1], linestyle="--",
                linewidth=2, label=f"Moy = {np.mean(cv_r2s):.4f}")
axes[1].set_ylabel("R²"); axes[1].set_title("R² par fold")
axes[1].legend(); axes[1].grid(True, axis="y")

fig.tight_layout()
save_fig(fig, "mlp_4_cross_validation.png")

✓ mlp_4_cross_validation.png


###8. Sauvegarde modèle

In [ ]:
model.save("mlp_affluence_final.keras")
import joblib
joblib.dump(scaler, "scaler_mlp.pkl")
print("\nModèle sauvegardé : mlp_affluence_final.keras")
print("Scaler sauvegardé : scaler_mlp.pkl")


▶ Modèle sauvegardé : mlp_affluence_final.keras
▶ Scaler sauvegardé : scaler_mlp.pkl


###9. Fonction Prédiction

In [ ]:
def predict_affluence(
    heure, mois, jour_semaine,
    temperature, pluie_1h,
    est_weekend, est_vacances,
    cat_jour,
    arret,
):
    """Prédit % de validations pour un cas donné."""
    features = np.array([[
        np.sin(2*np.pi*heure/24),
        np.cos(2*np.pi*heure/24),
        np.sin(2*np.pi*mois/12),
        np.cos(2*np.pi*mois/12),
        np.sin(2*np.pi*jour_semaine/7),
        np.cos(2*np.pi*jour_semaine/7),
        temperature,
        pluie_1h,
        int(pluie_1h > 0),
        int(pluie_1h > 2),
        est_weekend,
        est_vacances,
        le_cat.transform([cat_jour])[0],
        le_arrt.transform([arret])[0],
    ]])
    features_scaled = scaler.transform(features)
    pred = model.predict(features_scaled, verbose=0)[0][0]
    return round(float(pred), 2)


# Exemple prédiction (pour valider tous les pipelines) verif scaler + sanity check + lisibilité
exemple = predict_affluence(
    heure=8, mois=3, jour_semaine=0,
    temperature=12.0, pluie_1h=0.0,
    est_weekend=0, est_vacances=0,
    cat_jour="JOHV",
    arret="BECON"
)
print(f"\nExemple — Lundi 8h, 12°C, sec, BECON : {exemple}% de validations")

print("""

PIPELINE MLP TERMINÉ:

mlp_1_learning_curves.png
mlp_2_pred_vs_reel.png
mlp_3_residus.png
mlp_4_cross_validation.png
mlp_affluence_final.keras
scaler_mlp.pkl

""")


▶ Exemple — Lundi 8h, 12°C, sec, BECON : 13.07% de validations


PIPELINE MLP TERMINÉ:

mlp_1_learning_curves.png
mlp_2_pred_vs_reel.png
mlp_3_residus.png
mlp_4_cross_validation.png
mlp_affluence_final.keras
scaler_mlp.pkl                          


